# Lofi Generator

Using `"saikayala/jazz-ml-ready-midi"` from `kaggle` for training (more jazzy sounding).

## Prereqs

Let's get everything setup, first.

In [22]:
!apt-get update -qq && apt-get install -y lilypond > /dev/null

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [23]:
%pip install -q music21 kagglehub tqdm torch pydantic torch torchvision torchaudio

In [24]:
from music21 import environment

try:
    env = environment.UserSettings()
    env["lilypondPath"] = "/usr/bin/lilypond"
    env["graphicsMagickPath"] = "/usr/bin/display"
except Exception as e:
    print(f"Setup warning: {e}")

Setup warning: no preference: graphicsMagickPath


## Pydantic config

Will be helpful for type hinting, etc.

In [25]:
from pydantic import BaseModel, Field
from typing import Optional
import torch


class MusicConfig(BaseModel):
    dataset_handle: str = "saikayala/jazz-ml-ready-midi"
    extract_path: str = "./jazz_midi_extracted"

    sequence_length: int = 64
    embedding_dim: int = 256
    hidden_dim: int = 512
    num_layers: int = 2
    dropout: float = 0.3

    batch_size: int = 64
    learning_rate: float = 0.001
    epochs: int = 20
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

## Preprocessing

Turning MIDI files into tokens (key, time, notes).

In [ ]:
import glob
import kagglehub
import os
import tarfile
from music21 import converter, instrument, note, chord, stream, key
from tqdm.notebook import tqdm


class JazzPreprocessor:
    def __init__(self, config: MusicConfig):
        self.config = config
        self.vocab = []
        self.token_to_int = {}
        self.int_to_token = {}

    def setup_data(self):
        path = kagglehub.dataset_download(self.config.dataset_handle)
        archive_file = os.path.join(path, "Jazz-Midi.tar.xz")

        if not os.path.exists(self.config.extract_path):
            os.makedirs(self.config.extract_path)
            with tarfile.open(archive_file, "r:xz") as tar:
                for member in tqdm(tar.getmembers(), desc="Unpacking MIDI"):
                    tar.extract(member, path=self.config.extract_path)

        return glob.glob(
            os.path.join(self.config.extract_path, "**/*.mid"), recursive=True
        )

    def extract_metadata(self, score):
        try:
            k = score.analyze("key")
            key_token = f"KEY_{k.tonic.name}_{k.mode}"
            ts = score.getTimeSignatures()[0]
            ts_token = f"TIME_{ts.numerator}_{ts.denominator}"

            return [key_token, ts_token]
        except:
            return ["KEY_C_major", "TIME_4_4"]

    def tokenize_midi(self, file_path):
        try:
            score = converter.parse(file_path)
            meta = self.extract_metadata(score)
            tokens = meta

            parts = instrument.partitionByInstrument(score)
            tracks = parts.parts if parts else [score]

            for part in tracks:
                inst = part.partName if part.partName else "Piano"
                tokens.append(f"START_{inst}")
                for el in part.flatten().notesAndRests:
                    if isinstance(el, note.Note):
                        tokens.append(f"n_{el.pitch.nameWithOctave}")
                    elif isinstance(el, chord.Chord):
                        p_str = ".".join(sorted([n.nameWithOctave for n in el.pitches]))
                        tokens.append(f"c_{p_str}")
                    elif isinstance(el, note.Rest):
                        tokens.append("r")
                tokens.append(f"END_{inst}")
            return tokens
        except:
            return None

## Token to MIDI Decoder

This will have `lilypond` support.

In [27]:
class MidiConverter:
    @staticmethod
    def tokens_to_score(tokens: list[str], quarter_length: float = 0.5):
        score = stream.Score()
        current_part = None
        global_key, global_ts = None, None

        for t in tokens:
            if t.startswith("KEY_"):
                p = t.split("_")
                global_key = key.Key(p[1], p[2])
            elif t.startswith("TIME_"):
                p = t.split("_")
                global_ts = meter.TimeSignature(f"{p[1]}/{p[2]}")
            elif t.startswith("START_"):
                inst_name = t.replace("START_", "")
                current_part = stream.Part()
                try:
                    inst_obj = getattr(instrument, inst_name)()
                except:
                    inst_obj = instrument.Piano()
                current_part.insert(0, inst_obj)
                if global_key:
                    current_part.insert(0, global_key)
                if global_ts:
                    current_part.insert(0, global_ts)
            elif t.startswith("END_"):
                if current_part:
                    score.insert(0, current_part)
                    current_part = None
            elif current_part is not None:
                if t.startswith("n_"):
                    n = note.Note(t.replace("n_", ""))
                    n.duration.quarterLength = quarter_length
                    current_part.append(n)
                elif t.startswith("c_"):
                    pitches = t.replace("c_", "").split(".")
                    c = chord.Chord(pitches)
                    c.duration.quarterLength = quarter_length * 2
                    current_part.append(c)
                elif t == "r":
                    r = note.Rest()
                    r.duration.quarterLength = quarter_length
                    current_part.append(r)

        if current_part:
            score.insert(0, current_part)

        return score

In [28]:
def show_score(tokens, play=False):
    score = MidiConverter.tokens_to_score(tokens)
    if play:
        score.show("midi")
    else:
        score.show("lily.pdf")

In [ ]:
import torch.nn as nn


class JazzLSTM(nn.Module):
    def __init__(self, vocab_size, config: MusicConfig):
        super(JazzLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, config.embedding_dim)
        self.lstm = nn.LSTM(
            config.embedding_dim,
            config.hidden_dim,
            num_layers=config.num_layers,
            dropout=config.dropout,
            batch_first=True,
        )
        self.fc = nn.Linear(config.hidden_dim, vocab_size)

    def forward(self, x):
        embeds = self.embedding(x)
        lstm_out, _ = self.lstm(embeds)
        logits = self.fc(lstm_out[:, -1, :])
        return logits

## Training

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader


class JazzDataset(Dataset):
    def __init__(self, token_list, config: MusicConfig):
        self.config = config
        flat_tokens = [t for song in token_list for t in song]
        self.vocab = sorted(list(set(flat_tokens)))

        self.token_to_int = {t: i for i, t in enumerate(self.vocab)}
        self.int_to_token = {i: t for i, t in enumerate(self.vocab)}

        self.vocab_size = len(self.vocab)
        self.sequences = []
        self.targets = []

        for song in token_list:
            if len(song) <= config.sequence_length:
                continue
            for i in range(len(song) - config.sequence_length):
                seq_in = [
                    self.token_to_int[t] for t in song[i : i + config.sequence_length]
                ]
                target = self.token_to_int[song[i + config.sequence_length]]
                self.sequences.append(seq_in)
                self.targets.append(target)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.sequences[idx], dtype=torch.long),
            torch.tensor(self.targets[idx], dtype=torch.long),
        )

In [31]:
def train_model(model, dataset, config: MusicConfig):
    dataloader = DataLoader(dataset, batch_size=config.batch_size, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
    criterion = torch.nn.CrossEntropyLoss()

    model.to(config.device)
    model.train()

    epoch_pbar = tqdm(range(config.epochs), desc="Training Epochs")

    for epoch in epoch_pbar:
        model.train()
        total_loss = 0

        for x_batch, y_batch in dataloader:
            x_batch, y_batch = x_batch.to(config.device), y_batch.to(config.device)

            optimizer.zero_grad()
            output = model(x_batch)
            loss = criterion(output, y_batch)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(dataloader)
        epoch_pbar.set_postfix({"loss": f"{avg_loss:.4f}"})

## Generation

In [ ]:
import torch.nn.functional as F


def generate_music(model, dataset, seed_tokens, length=100, temperature=1.0):
    model.eval()
    current_seq = [dataset.token_to_int[t] for t in seed_tokens]
    generated_output = list(seed_tokens)

    for _ in tqdm(range(length), desc="Generating Jazz"):
        input_tensor = torch.tensor(
            [current_seq[-dataset.config.sequence_length :]]
        ).to(dataset.config.device)

        with torch.no_grad():
            logits = model(input_tensor)
            probs = F.softmax(logits / temperature, dim=-1)
            next_idx = torch.multinomial(probs, 1).item()

        generated_output.append(dataset.int_to_token[next_idx])
        current_seq.append(next_idx)

    return generated_output

## Testing

In [33]:
config = MusicConfig()
preprocessor = JazzPreprocessor(config)

midi_files = preprocessor.setup_data()

100%|██████████| 7.87M/7.87M [00:00<00:00, 69.7MB/s]

Extracting files...


Unpacking MIDI:   0%|          | 0/945 [00:00<?, ?it/s]

/tmp/ipykernel_8515/3522666761.py:24: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(member, path=self.config.extract_path)


In [ ]:
all_tokenized_songs = []
for f in tqdm(midi_files, desc="Processing MIDI Files"):
    tokens = preprocessor.tokenize_midi(f)
    if tokens:
        all_tokenized_songs.append(tokens)

print(f"Successfully processed {len(all_tokenized_songs)} songs.")

Processing MIDI Files:   0%|          | 0/943 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/music21/midi/translate.py:1943: TranslateWarning: Unable to decode lyrics from <music21.midi.MidiEvent LYRIC, track=0, data=b'na\x85 '> as utf-8
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/music21/midi/translate.py:922: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=2, data=b' \xc6\xc6\xc6\xc6\xc6\xc6\xc6\xc6'>; getting generic Instrument
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/music21/midi/translate.py:922: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=5, data=b' \xc6\xc6\xc6\xc6\xc6\xc6\xc6\xc6'>; getting generic Instrument
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/music21/midi/translate.py:1943: TranslateWarning: Unable to decode lyrics from <music21.midi.MidiEvent LYRIC, track=0, data=b'Intro\x85\x85\x85\x85\x85\x85\x85\x85\x85\x85\x85\x85\x85....\r\x00\x00'> as utf-8
  warnings.warn

## Saving Tokens

In [ ]:
import pickle

save_path = os.path.join(config.extract_path, "tokenized_data.pkl")

data_to_save = {
    "tokens": all_tokenized_songs,
    "config_dict": config.model_dump(),
}

with open(save_path, "wb") as f:
    pickle.dump(data_to_save, f)

print(f"Saved {len(all_tokenized_songs)} songs to {save_path}")

In [ ]:
from google.colab import files

# After your pickle.dump(data_to_save, f) block:
files.download(save_path)

In [ ]:
dataset = JazzDataset(all_tokenized_songs, config)
print(f"Total sequences for training: {len(dataset)}")

model = JazzLSTM(dataset.vocab_size, config)

In [ ]:
train_model(model, dataset, config)

In [ ]:
import random

random_song = random.choice(all_tokenized_songs)
seed_tokens = random_song[: config.sequence_length]

generated_tokens = generate_music(
    model,
    dataset,
    seed_tokens,
    length=256,
    temperature=0.9,
)

In [ ]:
show_score(generated_tokens)

In [ ]:
show_score(generated_tokens, play=True)

## Loading Tokens

In [ ]:
save_path = os.path.join(config.extract_path, "tokenized_data.pkl")

if os.path.exists(save_path):
    with open(save_path, "rb") as f:
        loaded_data = pickle.load(f)

    all_tokenized_songs = loaded_data["tokens"]
    config = MusicConfig(**loaded_data["config_dict"])

    print(f"Sucessfully loaded {len(all_tokenized_songs)} songs from disk.")
else:
    print("No saved data found. You'll need to run the preprocessor.")